# FPL xPts, xG and xA projection runner

Run this notebook from the repository root in Google Colab. It fetches live FPL and Understat data, uses the uploaded Elevenify team goals/clean-sheet CSV when present, and writes the legacy point-projection CSVs with unchanged filenames and column order.

In [ ]:
from pathlib import Path
import os
import shutil
import sys
import pandas as pd

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB and not (Path.cwd() / 'src' / 'fpl_xpts').exists():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

search_roots = [Path.cwd(), Path('/content'), Path('/content/drive/MyDrive')]
candidates = []
for root in search_roots:
    if root.exists():
        candidates.extend([root, *[p for p in root.iterdir() if p.is_dir()]])
        for child in [p for p in root.iterdir() if p.is_dir()]:
            try:
                candidates.extend([p for p in child.iterdir() if p.is_dir()])
            except Exception:
                pass
ROOT = next((p for p in candidates if (p / 'src' / 'fpl_xpts').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Upload/open the repo folder so src/fpl_xpts exists, then run again.')
os.chdir(ROOT)
SRC = ROOT / 'src'
sys.path.insert(0, str(SRC.resolve()))

# Keep the uploaded Elevenify season-long CSV in the repo root. It overrides the fallback team goal/CS model for GW37-GW38.
# Optional fallback if you do not have the Elevenify file:
# os.environ['ODDS_API_KEY'] = 'paste-your-the-odds-api-key-here'

from fpl_xpts.config import AppConfig
from fpl_xpts.legacy_export import (
    fixture_player_week,
    form_weighting_audit,
    qc_tables,
    shot_profile_audit,
    six_week_totals,
    weekly_player_week,
)
from fpl_xpts.pipeline import run_live_projection

OUT_DIR = Path('outputs/legacy_live')
OUT_DIR.mkdir(parents=True, exist_ok=True)
MINUTES_INPUT = Path('player_minutes_inputs_gw37_to_38.csv') if Path('player_minutes_inputs_gw37_to_38.csv').exists() else Path('player_minutes_inputs.csv')
CONFIG = AppConfig(
    n_sim=10_000,
    projection_start_gw=37,
    projection_end_gw=38,
    use_market_odds=True,
    use_elevenify_projection_file=True,
    use_fpl_player_history=True,
    use_player_minutes_input_file=True,
    player_minutes_input_path=MINUTES_INPUT,
    write_player_minutes_input_template=True,
    overwrite_player_minutes_input_template=False,
    use_external_team_projection_files=False,
    use_understat_profiles=True,
    include_big_chance_profiles=True,
    understat_season=None,
)
CONFIG

In [ ]:
live = run_live_projection(CONFIG, include_mc=False)
fixture_df = fixture_player_week(live['player_fixture'], live['players'], live['teams'])
weekly_df = weekly_player_week(fixture_df)
totals_df = six_week_totals(weekly_df)
qc_week, qc_fixture = qc_tables(fixture_df)
form_audit = form_weighting_audit(live['players'], live['teams'])
shot_audit = shot_profile_audit(live.get('shot_profiles', pd.DataFrame()))

outputs = {
    'fixture_player_week.csv': fixture_df,
    'weekly_player_week.csv': weekly_df,
    'six_week_totals.csv': totals_df,
    'top50_p1_ga_by_week.csv': weekly_df.sort_values(['week', 'P1_GA'], ascending=[True, False]).groupby('week', group_keys=False).head(50).reset_index(drop=True),
    'top50_xga_by_week.csv': weekly_df.sort_values(['week', 'xGA_exp'], ascending=[True, False]).groupby('week', group_keys=False).head(50).reset_index(drop=True),
    'qc_team_week.csv': qc_week,
    'qc_team_week_fixture.csv': qc_fixture,
    'form_weighting_audit.csv': form_audit,
    'shot_profile_audit.csv': shot_audit,
}
for name, df in outputs.items():
    df.to_csv(OUT_DIR / name, index=False, float_format='%.6f')
archive = shutil.make_archive(str(OUT_DIR), 'zip', OUT_DIR)
sorted(outputs), archive

In [ ]:
weekly_df.loc[weekly_df['GW'].eq(37), ['player', 'team', 'Pos', 'mins', 'xG_scaled', 'xA_scaled', 'xPts', 'fixtures_in_week']].head(25)

In [ ]:
shot_audit[['player', 'team', 'understat_shots90', 'understat_xG_per_shot', 'understat_chances_created90', 'understat_xA_per_chance', 'understat_big_chance_received90', 'understat_big_chance_created90']].head(25)